# Historical storm analysis — pipeline, evaluation and manuscript display items

Runs the probabilistic damage-and-recovery pipeline on 10 historical Gulf/Atlantic
storms (each with frequency = 1) and produces the historical display items of the
manuscript:

**Main text**
- `figures/hist_AL132020_4panel_focus.png` (Hurricane Laura, 2020)
- `tables/table_laura_main.tex` (four Louisiana parishes)

**Supplementary Information**
- `figures/hist_AL142018_4panel_focus.png` (Hurricane Michael, 2018)
- `tables/table_laura_SI.tex` (all damaged counties, Laura)
- `tables/table_damage_distribution.tex` (FEMA-primary damage evaluation)
- `tables/table_damage_normalized.tex` (per-exposure-unit robustness)
- `tables/table_scaling_robustness.tex` (rain/surge scaling insensitivity)
- `tables/table_huang_thresholds.tex` (Huang et al. 2025 filter sensitivity)

**Steps**
1. Inspect historical wind fields (external input; skipped if absent)
2. Build CLIMADA Hazard (`frequency = 1`) from per-storm wind fields (cached: `data/hazard/gori_historical.hdf5`)
3. Historical scaling NPZ from Dmat (cached: `data/scaling_relative_historical.npz`)
4. Impacts per coastal state with historical scaling (cached: `data/impact_historical/per_event/`)
5. Recovery potential via pyrecodes-light (`modules/recovery_utils.py`; cached: `data/recovery_historical/recovery_potential.csv`)
6. Manuscript figures (Michael and Laura focus maps)
7. Evaluation against OpenFEMA IA damage and Huang et al. (2025) recovery; manuscript tables

Steps 2–4 need CLIMADA and the external wind-field inputs; they are skipped
automatically when the cached outputs listed above are present, so Steps 5–7
(and all display items) reproduce from the committed repository data alone
(plus `climada` for reading the cached hazard in the focus figures).

In [ ]:
from __future__ import annotations

import os, sys, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
from scipy.stats import spearmanr

# -- repo root: works whether the kernel CWD is notebooks/ or the repo root --
_cwd = Path('.').resolve()
if (_cwd / 'data').exists():
    REPO = _cwd               # already at repo root
else:
    REPO = _cwd.parent        # one level up (typical when run from notebooks/)
    if not (REPO / 'data').exists():
        raise RuntimeError(f"Cannot locate repo root from {_cwd}")

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
os.chdir(REPO)   # ensure relative paths in modules work

warnings.filterwarnings('ignore', category=UserWarning)
print(f'REPO = {REPO}')


## Paths and target storms

In [ ]:
# ---- CLIMADA system directory (external inputs; override via CLIMADA_DATA env var) ----
CLIMADA_DATA = Path(os.environ.get('CLIMADA_DATA', Path.home() / 'climada' / 'data'))

# ---- Historical wind-field inputs (external; only needed to (re)build the hazard) ----
HIST_ROOT   = CLIMADA_DATA / 'hazard' / 'tropical_cyclone' / 'gori' / 'historical'
BLENDED_DIR = HIST_ROOT   # per-storm ATCF-named .mat files sit directly here
TRACK_MAT   = HIST_ROOT / 'Besttrack_GOM_1940_TS_trk100_corrected2.mat'

# ---- Repo data files ----
DATA_DIR          = REPO / 'data'
COUNTY_REGION_CSV = DATA_DIR / 'county_region.csv'
COUNTIES_SHP      = DATA_DIR / 'US_counties.shp'
DMAT_CSV          = DATA_DIR / 'Dmat_region_all.csv'
HUANG_CSV         = DATA_DIR / 'huang_recovery_by_county_event.csv'
PERMITS_CSV       = DATA_DIR / 'selected_states_counties_with_permits.csv'

# ---- Per-state exposure HDF5 (repo layout first, CLIMADA directory as fallback) ----
EXP_DIR = DATA_DIR / 'exposure' / 'states'
if not EXP_DIR.exists():
    EXP_DIR = CLIMADA_DATA / 'exposure' / 'states'

# ---- Output paths ----
HAZ_HDF5          = DATA_DIR / 'hazard' / 'gori_historical.hdf5'
SCALING_NPZ_HIST  = DATA_DIR / 'scaling_relative_historical.npz'
IMPACT_DIR_HIST   = DATA_DIR / 'impact_historical'
RECOVERY_DIR_HIST = DATA_DIR / 'recovery_historical'
FIGURES_DIR       = REPO / 'figures'
TABLES_DIR        = REPO / 'tables'

for d in [HAZ_HDF5.parent, IMPACT_DIR_HIST, RECOVERY_DIR_HIST, FIGURES_DIR, TABLES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ---- Target storms ----
TARGET_STORMS = [
    dict(atcf='AL142018', name='Michael',  year=2018, role='comparison'),
    dict(atcf='AL132020', name='Laura',    year=2020, role='comparison'),
    dict(atcf='AL092017', name='Harvey',   year=2017, role='comparison'),
    dict(atcf='AL112017', name='Irma',     year=2017, role='comparison'),
    dict(atcf='AL092012', name='Isaac',    year=2012, role='comparison'),
    dict(atcf='AL092016', name='Hermine',  year=2016, role='comparison'),
    dict(atcf='AL162017', name='Nate',     year=2017, role='comparison'),   # may be missing
    dict(atcf='AL122005', name='Katrina',  year=2005, role='illustration'), # not in Huang
    dict(atcf='AL092021', name='Ida',      year=2021, role='illustration'), # post-Dmat, wind-only
    dict(atcf='AL092022', name='Ian',      year=2022, role='illustration'), # post-Dmat, wind-only
]

# Storms without Dmat rain/surge coverage (set Scaling = 1)
WIND_ONLY_ATCF = ['AL092021', 'AL092022']   # Ida and Ian (Dmat ends 2020)

# Storms in Huang dataset used for evaluation
# Nate (AL162017) excluded: missing from historical wind field set
HUANG_ATCF_IDS = [
    'AL142018', 'AL132020', 'AL092017', 'AL112017',
    'AL092012', 'AL092016',
]

TARGET_ATCF = [s['atcf'] for s in TARGET_STORMS]
STORM_YEARS = {s['atcf']: s['year'] for s in TARGET_STORMS}
STORM_LABEL = {s['atcf']: f"{s['name']} ({s['year']})" for s in TARGET_STORMS}

print(f'Target ATCF IDs: {TARGET_ATCF}')
print(f'Wind-only (no Dmat): {WIND_ONLY_ATCF}')
print(f'Huang comparison: {HUANG_ATCF_IDS}')


## Step 1 — Inspect historical wind fields and identify missing storms

Needs the external blended wind-field `.mat` files; skipped when they are absent
(the cached hazard HDF5 is used instead in Step 2).

In [ ]:
if BLENDED_DIR.exists():
    available_mats = {fp.stem for fp in BLENDED_DIR.glob('*.mat') if fp.stem != TRACK_MAT.stem}
    print(f'Historical wind field .mat files found: {len(available_mats)}')
    print(sorted(available_mats))
    print()
    PRESENT_ATCF = [a for a in TARGET_ATCF if a in available_mats]
    MISSING_ATCF = [a for a in TARGET_ATCF if a not in available_mats]
    for s in TARGET_STORMS:
        status = 'ok' if s['atcf'] in available_mats else 'MISSING'
        print(f"  {s['atcf']}  {s['name']:<10} {s['year']}  {status}")
    print(f'\nStorms PRESENT ({len(PRESENT_ATCF)}): {PRESENT_ATCF}')
    print(f'Storms MISSING ({len(MISSING_ATCF)}): {MISSING_ATCF}')
else:
    print(f'{BLENDED_DIR} not found — external wind fields unavailable.')
    print('Step 2 will fall back to the cached hazard HDF5.')
    PRESENT_ATCF, MISSING_ATCF = [], []


## Step 2 — Build CLIMADA Hazard from historical wind fields

Loads the cached `data/hazard/gori_historical.hdf5` when present; otherwise
builds it from the blended wind fields (requires the external `.mat` inputs).

In [ ]:
from climada.hazard import Hazard

if HAZ_HDF5.exists():
    print(f'Loading cached hazard from {HAZ_HDF5}')
    haz_hist = Hazard.from_hdf5(str(HAZ_HDF5))
    missing_blended = MISSING_ATCF   # from Step 1 (empty when skipped)
else:
    print('Building hazard from historical wind fields ...')
    from modules.hazard_utils import load_tc_hazard_from_blended_mats
    haz_hist, missing_blended = load_tc_hazard_from_blended_mats(
        blended_dir     = BLENDED_DIR,
        track_mat_path  = TRACK_MAT,
        atcf_ids        = TARGET_ATCF,
        resolution_deg  = 0.25,   # must match probabilistic hazard
        pad_deg         = 1.0,
    )
    haz_hist.write_hdf5(str(HAZ_HDF5))
    print(f'Saved hazard -> {HAZ_HDF5}')

loaded_atcf = list(haz_hist.event_name)
print(f'Hazard events loaded  : {len(loaded_atcf)}')
print(f'Events               : {loaded_atcf}')
print(f'Centroids            : {haz_hist.centroids.size:,}')
print(f'Frequency (all 1?)   : {np.unique(haz_hist.frequency)}')
print(f'Missing wind fields  : {missing_blended}')


## Step 3 — Historical scaling NPZ

Build W (from hazard), R and S (from Dmat), compute Scaling via the same
relative-contribution formulation as the probabilistic pipeline.
Ida (2021) and Ian (2022) are forced to Scaling = 1 (wind-only, no Dmat).
Loads the cached `data/scaling_relative_historical.npz` when present.

In [ ]:
haz_atcf_ids = list(haz_hist.event_name)
haz_years    = [STORM_YEARS[a] for a in haz_atcf_ids]
hist_wind_only = [a for a in haz_atcf_ids if a in WIND_ONLY_ATCF]
print(f'Loaded events: {haz_atcf_ids}')
print(f'Wind-only (Scaling=1): {hist_wind_only}')

if SCALING_NPZ_HIST.exists():
    print(f'\nLoading cached historical scaling NPZ from {SCALING_NPZ_HIST}')
else:
    from modules.scaling_utils import build_historical_scaling_npz

    county_region_df = pd.read_csv(COUNTY_REGION_CSV)
    if 'region' not in county_region_df.columns and 'region_str' in county_region_df.columns:
        county_region_df = county_region_df.rename(columns={'region_str': 'region'})
    dmat_df = pd.read_csv(DMAT_CSV)
    print(f'Counties: {len(county_region_df)}  |  Dmat rows: {len(dmat_df)}  '
          f'(years {dmat_df["year"].min()}-{dmat_df["year"].max()})')

    W_hist, R_hist, S_hist, match_report = build_historical_scaling_npz(
        haz               = haz_hist,
        event_atcf_ids    = haz_atcf_ids,
        event_years       = haz_years,
        county_region_df  = county_region_df,
        dmat_df           = dmat_df,
        counties_shp_path = COUNTIES_SHP,
        out_path          = SCALING_NPZ_HIST,
        wind_only_atcf_ids= hist_wind_only,
    )
    # Windmax match diagnostics (Dmat storm identification)
    print(f'\nDmat match records: {len(match_report)}')
    if not match_report.empty:
        print(f'  Residual (m/s) — mean: {match_report["residual"].mean():.2f},  '
              f'p50: {match_report["residual"].median():.2f},  '
              f'max: {match_report["residual"].max():.2f}')
        n_amb = int(match_report['ambiguous'].sum())
        print(f'  Ambiguous matches (windmax gap < 1 m/s): {n_amb}/{len(match_report)}')

# ---- Inspect scaling values ----
_sc = np.load(SCALING_NPZ_HIST, allow_pickle=True)
Scaling_hist = _sc['Scaling']
print(f'\nScaling matrix shape: {Scaling_hist.shape}')
print('Per-event Scaling summary (mean of values > 1):')
for e_idx, aid in enumerate(haz_atcf_ids):
    sv = Scaling_hist[e_idx, :]
    above = sv[sv > 1.0]
    wof = '(wind-only)' if aid in hist_wind_only else ''
    if above.size > 0:
        print(f'  {aid}: counties with Scaling>1 = {above.size}, '
              f'mean = {above.mean():.2f}, max = {above.max():.2f} {wof}')
    else:
        print(f'  {aid}: all Scaling = 1.0 {wof}')


## Step 4 — Impact: wind damage + historical scaling, per state

Skipped when `data/impact_historical/per_event/` already holds the per-event CSVs.
Recomputing requires the per-state exposure HDF5 files (see README).

In [ ]:
per_event_dir = IMPACT_DIR_HIST / 'per_event'
per_event_dir.mkdir(parents=True, exist_ok=True)

existing_csvs = list(per_event_dir.glob('*.csv'))
if existing_csvs:
    print(f'Found {len(existing_csvs)} existing per-event CSVs in {per_event_dir}')
    print('Skipping impact computation. Delete the directory to recompute.')
else:
    from climada.entity.exposures import Exposures
    from climada.engine import ImpactCalc
    from modules.impfunc_utils import IMPF_SET_TC_CAPRA, DICT_PAGER_TCIMPF_CAPRA
    from modules.impact_utils import export_state_and_county_results_all_events

    state_files = sorted(EXP_DIR.glob('*_exposure.hdf5'))
    STATE_NAMES = [fp.stem.replace('_exposure', '') for fp in state_files]
    print(f'Found {len(STATE_NAMES)} state exposures in {EXP_DIR}')
    print(f'Running ImpactCalc for {len(STATE_NAMES)} states x {len(haz_atcf_ids)} events ...')

    for state_name in STATE_NAMES:
        exp_file = EXP_DIR / f'{state_name}_exposure.hdf5'
        print(f'  {state_name} ...', end=' ', flush=True)
        try:
            exp = Exposures.from_hdf5(str(exp_file))
        except Exception as e:
            print(f'FAILED to load exposure: {e}')
            continue

        exp.gdf['impf_TC'] = exp.gdf.apply(
            lambda row: DICT_PAGER_TCIMPF_CAPRA[row.StructureType], axis=1
        )

        try:
            imp = ImpactCalc(exp, IMPF_SET_TC_CAPRA, haz_hist).impact(save_mat=True)
        except Exception as e:
            print(f'FAILED ImpactCalc: {e}')
            continue

        export_state_and_county_results_all_events(
            exp               = exp,
            imp               = imp,
            scaling_npz_path  = str(SCALING_NPZ_HIST),
            county_region_path= str(COUNTY_REGION_CSV),
            out_dir           = str(IMPACT_DIR_HIST),
            k                 = 1.0,
            scale_mode        = 'compound',
            lower_threshold   = 0.005,
        )
        print('done')

print('Per-event CSVs available:', len(list(per_event_dir.glob('*.csv'))))


In [ ]:
# ---- Sanity check: inspect a sample impact CSV ----
sample_csvs = sorted(per_event_dir.glob('*.csv'))
if sample_csvs:
    df_sample = pd.read_csv(sample_csvs[0])
    print(f'Sample: {sample_csvs[0].name}')
    print(df_sample.head(5).to_string())
    print()
    # Summary: which event_names appear?
    all_csvs = pd.concat([pd.read_csv(f) for f in sample_csvs], ignore_index=True)
    print('Events in per_event CSVs:', sorted(all_csvs['event_name'].unique()))
    print('County-events:', len(all_csvs))


## Step 5 — Recovery potential (pyrecodes-light)

Uses `modules.recovery_utils.compute_recovery_potential` (the same function the
probabilistic pipeline calls through `scripts/run_pyrecodes_light.py`).

In [ ]:
RECOVERY_CSV = RECOVERY_DIR_HIST / 'recovery_potential.csv'

if RECOVERY_CSV.exists():
    print(f'Loading cached recovery potential from {RECOVERY_CSV}')
    recovery = pd.read_csv(RECOVERY_CSV)
else:
    from modules.recovery_utils import compute_recovery_potential
    recovery = compute_recovery_potential(
        per_event_dir = per_event_dir,
        permit_file   = PERMITS_CSV,
        out_csv       = RECOVERY_CSV,
    )

print(f'Recovery rows: {len(recovery):,}')
print(f'Events: {sorted(recovery["event_name"].unique())}')
print(recovery.head(5).to_string())


## Step 6 — Manuscript figures: Michael (2018) and Laura (2020) focus maps

Four-panel maps (`hist_{ATCF}_4panel_focus.png`): wind footprint overview with the
regional inset marked, then regional detail of WUA, CC and recovery potential.
WUA = Weighted Units Affected (sum of units_DSk x tau_k, tau = {DS1: 1, DS2: 1, DS3: 3, DS4: 6}).
CC = Construction Capacity (average monthly building permits over the past 12 months).
Laura is Figure 1 of the main text; Michael appears in the SI.

In [ ]:
from matplotlib.colors import LogNorm

# ---- Style constants matching notebooks/probabilistic_analysis.ipynb ----
W_DOUBLE = 5.91   # 15.0 cm, double column
MAP_XLIM = (-107, -65)
MAP_YLIM = (24, 48)
NO_DATA_COLOR    = "#bab9b9"
COUNTY_EDGECOLOR = "#aaaaaa"
COUNTY_LINEWIDTH = 0.1
STATE_EDGECOLOR  = "#444444"
STATE_LINEWIDTH  = 0.3
CBAR_FS          = 8   # uniform colorbar fontsize

COASTAL_STATE_FIPS = [
    "01", "09", "10", "12", "13", "22", "23", "24", "25", "28",
    "33", "34", "36", "37", "42", "44", "45", "48", "51",
]

plt.rcParams.update({
    "font.family":     "sans-serif",
    "font.sans-serif": ["Helvetica", "Arial", "DejaVu Sans"],
})

# ---- Load spatial base layers ----
counties_raw = gpd.read_file(str(COUNTIES_SHP))
if counties_raw.crs is None:
    counties_raw = counties_raw.set_crs("EPSG:4326")
coastal_counties = counties_raw[counties_raw["STATEFP"].isin(COASTAL_STATE_FIPS)].copy()
coastal_counties["GEOID"] = (coastal_counties["STATEFP"] + coastal_counties["COUNTYFP"]).str.zfill(5)
state_gdf = coastal_counties.dissolve(by="STATEFP").reset_index()
print(f"Coastal counties: {len(coastal_counties)}, states: {len(state_gdf)}")


def _choropleth_panel(gdf, state_gdf, ax, col, cmap, cbar_label,
                      invert_cbar=False, end_labels=None,
                      xlim=None, ylim=None):
    """Single choropleth panel matching manuscript figS2 layout.

    end_labels: ("Low", "High") tuple for the inverted recovery colorbar.
    xlim/ylim: override map extent (defaults to full US coast).
    """
    _xlim = xlim if xlim is not None else MAP_XLIM
    _ylim = ylim if ylim is not None else MAP_YLIM

    tmp = gdf[[col, "geometry"]].copy()
    tmp.loc[~(tmp[col] > 0), col] = np.nan
    valid = tmp[col].dropna()
    norm = None
    if len(valid) > 0:
        vmin, vmax = valid.min(), valid.max()
        if np.isfinite(vmin) and np.isfinite(vmax) and vmin > 0:
            norm = LogNorm(vmin=vmin / 2, vmax=vmax)

    cax = ax.inset_axes([0.125, -0.12, 0.75, 0.05])
    tmp.plot(
        column=col, cmap=cmap, norm=norm,
        edgecolor=COUNTY_EDGECOLOR, linewidth=COUNTY_LINEWIDTH,
        legend=True, ax=ax, cax=cax,
        legend_kwds={"orientation": "horizontal"},
        missing_kwds={"color": NO_DATA_COLOR,
                      "edgecolor": COUNTY_EDGECOLOR,
                      "linewidth": COUNTY_LINEWIDTH},
    )
    state_gdf.plot(ax=ax, facecolor="none",
                   edgecolor=STATE_EDGECOLOR, linewidth=STATE_LINEWIDTH)
    ax.set_xlim(_xlim)
    ax.set_ylim(_ylim)
    ax.set_aspect("equal")
    ax.axis("off")

    # Colorbar: uniform fontsize, label below the bar
    cax.tick_params(labelsize=CBAR_FS)
    cax.tick_params(which="minor", length=0)
    if invert_cbar:
        cax.invert_xaxis()
        cax.set_xticks([])
        # "Low … label … High" placed below the bar as one xlabel string
        _end = end_labels if end_labels is not None else ("Low", "High")
        cax.set_xlabel(f"{_end[0]}     {cbar_label}     {_end[1]}",
                       fontsize=CBAR_FS, labelpad=2)
    else:
        cax.set_xlabel(cbar_label, fontsize=CBAR_FS, labelpad=2)


RECOVERY_WEIGHTS = {1: 1, 2: 1, 3: 3, 4: 6}


In [ ]:
import matplotlib.patches as mpatches

# ---- Focus: Michael and Laura — 4-panel maps ----
# a: Wind footprint overview (19 coastal states) with red box marking the inset extent
# b–d: Regional detail (WUA, CC, Recovery potential)

FOCUS_ATCF = ['AL142018', 'AL132020']
PAD_DEG    = 1.5
WIND_THRESH = 17.5   # m/s (tropical-storm strength)

for atcf_id in FOCUS_ATCF:
    if atcf_id not in haz_atcf_ids:
        print(f"{atcf_id}: not in this run — skipping")
        continue

    event_csvs = sorted(per_event_dir.glob(f"*_{atcf_id}.csv"))
    if not event_csvs:
        print(f"  {atcf_id}: no impact CSVs — skipping")
        continue

    impact_df = pd.concat([pd.read_csv(f) for f in event_csvs], ignore_index=True)
    impact_df["fips"] = impact_df["fips"].astype(str).str.zfill(5)
    impact_df["weighted_damage"] = sum(
        impact_df[f"units_DS{k}_scaled"] * RECOVERY_WEIGHTS[k] for k in [1, 2, 3, 4]
    )

    rec_ev = recovery[recovery["event_name"] == atcf_id].copy()
    rec_ev["fips"] = rec_ev["fips"].astype(str).str.zfill(5)
    rec_ev["cc"] = rec_ev["reconstruction_capacity"]   # already permits/month
    rec_ev = rec_ev.rename(columns={"recovery_potential_months": "recovery_months"})

    ev_df = impact_df[["fips", "weighted_damage"]].merge(
        rec_ev[["fips", "cc", "recovery_months"]], on="fips", how="outer"
    )

    gdf = coastal_counties.merge(ev_df, left_on="GEOID", right_on="fips", how="left")
    for col in ("weighted_damage", "cc", "recovery_months"):
        gdf.loc[~(gdf[col] > 0), col] = np.nan
    # CC only where the county has non-zero damage
    gdf.loc[gdf["weighted_damage"].isna(), "cc"] = np.nan

    affected = gdf[gdf["weighted_damage"] > 0]
    if affected.empty:
        print(f"  {atcf_id}: no affected counties — skipping")
        continue
    b = affected.total_bounds
    crop_xlim = (b[0] - PAD_DEG, b[2] + PAD_DEG)
    crop_ylim = (b[1] - PAD_DEG, b[3] + PAD_DEG)

    n_aff = int((gdf["weighted_damage"] > 0).sum())
    print(f"{STORM_LABEL.get(atcf_id, atcf_id)}: {n_aff} counties  |  "
          f"lon {crop_xlim[0]:.1f}–{crop_xlim[1]:.1f}, lat {crop_ylim[0]:.1f}–{crop_ylim[1]:.1f}")

    # ---- Wind footprint ----
    ev_idx = list(haz_hist.event_name).index(atcf_id)
    wind   = haz_hist.intensity[ev_idx, :].toarray().ravel()
    lons_h = haz_hist.centroids.lon
    lats_h = haz_hist.centroids.lat
    wmask  = wind > WIND_THRESH

    # ---- 4-panel figure ----
    fig, axes = plt.subplots(1, 4, figsize=(W_DOUBLE * 4 / 3, 3.3))

    # Panel a — wind overview + red inset box
    ax0 = axes[0]
    coastal_counties.plot(ax=ax0, color=NO_DATA_COLOR,
                          edgecolor=COUNTY_EDGECOLOR, linewidth=COUNTY_LINEWIDTH)
    state_gdf.plot(ax=ax0, facecolor="none",
                   edgecolor=STATE_EDGECOLOR, linewidth=STATE_LINEWIDTH)
    if wmask.any():
        sc = ax0.scatter(lons_h[wmask], lats_h[wmask], c=wind[wmask],
                         s=2, cmap="YlOrRd", vmin=WIND_THRESH, vmax=65,
                         rasterized=True, zorder=3)
        cax0 = ax0.inset_axes([0.125, -0.12, 0.75, 0.05])
        plt.colorbar(sc, cax=cax0, orientation="horizontal")
        cax0.tick_params(labelsize=CBAR_FS)
        cax0.tick_params(which="minor", length=0)
        cax0.set_xlabel("Wind speed [m s⁻¹]", fontsize=CBAR_FS, labelpad=2)
    # Red box marking the regional inset
    ax0.plot(
        [crop_xlim[0], crop_xlim[1], crop_xlim[1], crop_xlim[0], crop_xlim[0]],
        [crop_ylim[0], crop_ylim[0], crop_ylim[1], crop_ylim[1], crop_ylim[0]],
        color="red", linewidth=1.5, zorder=5,
    )
    ax0.set_xlim(MAP_XLIM)
    ax0.set_ylim(MAP_YLIM)
    ax0.set_aspect("equal")
    ax0.axis("off")
    ax0.text(-0.01, 0.98, "a)", transform=ax0.transAxes,
             ha="left", va="top", fontsize=10, fontweight="bold")

    # Panels b–d — regional detail
    panels = [
        ("weighted_damage", "cividis",   "WUA [units]",          False, None),
        ("cc",              "Greens",    "CC [permits month⁻¹]", False, None),
        ("recovery_months", "Purples_r", "Recovery potential",    True,  ("Low", "High")),
    ]
    for ax, (col, cmap, cbar_label, inv, end_lbls), lbl in zip(axes[1:], panels, ["b", "c", "d"]):
        _choropleth_panel(gdf, state_gdf, ax, col, cmap, cbar_label,
                          invert_cbar=inv, end_labels=end_lbls,
                          xlim=crop_xlim, ylim=crop_ylim)
        ax.text(0.01, 0.98, f"{lbl})", transform=ax.transAxes,
                ha="left", va="top", fontsize=10, fontweight="bold")

    plt.tight_layout(pad=0.5)
    out_fp = FIGURES_DIR / f"hist_{atcf_id}_4panel_focus.png"
    fig.savefig(out_fp, dpi=300, bbox_inches="tight")
    print(f"  Saved → {out_fp}")
    plt.show()


## Step 7 — Evaluation and manuscript tables

Evaluation of the modelled damage distribution against FEMA-verified housing
damage (OpenFEMA Individual Assistance, primary anchor; `data/fema_ia/`, see
`scripts/fetch_openfema_damage.py`) and the Huang et al. (2025) building sample
(secondary anchor), plus the county tables for Hurricane Laura. Writes the five
manuscript tables to `tables/`.

In [ ]:
# ---- Pooled Spearman correlations vs Huang et al. (2025), primary filter n_changed >= 10 ----
# Reported in the main text; the full threshold sensitivity is in table_huang_thresholds below.
huang = pd.read_csv(HUANG_CSV)
huang['GEOID'] = huang['GEOID'].astype(str).str.zfill(5)

rec_eval = recovery.copy()
rec_eval['fips5'] = rec_eval['fips'].astype(str).str.zfill(5)
joined = huang.merge(
    rec_eval[['event_name', 'fips5', 'recovery_potential_months', 'reconstruction_capacity']],
    left_on=['GEOID', 'atcf_id'], right_on=['fips5', 'event_name'], how='inner',
)
eval_df = joined[joined['atcf_id'].isin(HUANG_ATCF_IDS) & (joined['n_changed'] >= 10)]

CORR_PAIRS = [
    ('reconstruction_capacity',   'rebuild_rate',     '> 0'),
    ('recovery_potential_months', 'recovery_score',   '< 0'),
    ('recovery_potential_months', 'abandonment_rate', '> 0'),
]
print(f'County-events after filter (comparison storms, n_changed >= 10): {len(eval_df)}\n')
print(f"{'pair':<58} {'n':>4} {'rho':>7} {'p':>8}")
for x, y, sign in CORR_PAIRS:
    sub = eval_df[[x, y]].dropna()
    rho, p = spearmanr(sub[x], sub[y])
    print(f'{x + " vs " + y:<58} {len(sub):>4} {rho:>+7.2f} {p:>8.3f}   (expected {sign})')


In [ ]:
# ---- Table inputs / outputs ----
PER_EVENT_DIR = IMPACT_DIR_HIST / 'per_event'
COUNTIES_DBF  = DATA_DIR / 'US_counties.dbf'
FEMA_CSV      = DATA_DIR / 'fema_ia' / 'fema_damage_by_county.csv'
OUT_DIR       = TABLES_DIR

TAU = {1: 1, 2: 1, 3: 3, 4: 6}  # repair-time weights (months) per damage state

STORMS = {  # ATCF -> (name, year)
    "AL122005": ("Katrina", 2005),
    "AL092012": ("Isaac", 2012),
    "AL092016": ("Hermine", 2016),
    "AL092017": ("Harvey", 2017),
    "AL112017": ("Irma", 2017),
    "AL162017": ("Nate", 2017),
    "AL142018": ("Michael", 2018),
    "AL132020": ("Laura", 2020),
    "AL092021": ("Ida", 2021),
    "AL092022": ("Ian", 2022),
}


def load_event(atcf: str) -> pd.DataFrame | None:
    files = sorted(PER_EVENT_DIR.glob(f"aggregated_*_{atcf}.csv"))
    if not files:
        return None
    df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
    df["fips"] = df["fips"].astype(str).str.split(".").str[0].str.zfill(5)
    df["wua_raw"] = sum(df[f"units_DS{k}_raw"] * TAU[k] for k in TAU)
    df["wua_scaled"] = sum(df[f"units_DS{k}_scaled"] * TAU[k] for k in TAU)
    return df


def county_names() -> dict:
    """GEOID -> (name, state_abbr) from the counties shapefile DBF."""
    state_abbr = {
        "01": "AL", "12": "FL", "13": "GA", "22": "LA", "28": "MS",
        "45": "SC", "48": "TX",
    }
    try:
        import geopandas as gpd

        gdf = gpd.read_file(COUNTIES_DBF.with_suffix(".shp"))
        return {
            str(g): (n, state_abbr.get(str(s), str(s)))
            for g, n, s in zip(gdf["GEOID"], gdf["NAME"], gdf["STATEFP"])
        }
    except ImportError:
        from dbfread import DBF

        return {
            str(r["GEOID"]): (r["NAME"], state_abbr.get(str(r["STATEFP"]), ""))
            for r in DBF(COUNTIES_DBF, encoding="utf-8")
        }


def fmt_int(x: float) -> str:
    return f"{int(round(x)):,}"


### Laura county tables — `table_laura_main.tex` (main text), `table_laura_SI.tex` (SI)

In [ ]:
# ================================================================ Laura tables
recovery = pd.read_csv(RECOVERY_CSV)
recovery["fips"] = recovery["fips"].astype(str).str.zfill(5)
names = county_names()

laura = load_event("AL132020").merge(
    recovery.loc[
        recovery.event_name == "AL132020",
        ["fips", "reconstruction_capacity", "recovery_potential_months"],
    ],
    on="fips",
    how="left",
)
laura = laura[laura["wua_scaled"] > 0].sort_values("wua_scaled", ascending=False)
laura["county"] = laura["fips"].map(lambda f: names.get(f, (f, ""))[0])
laura["state"] = laura["fips"].map(lambda f: names.get(f, (f, ""))[1])

# ---- main-text table: the four counties agreed with SM
MAIN_FIPS = ["22019", "22011", "22079", "22115"]
rows = []
for f in MAIN_FIPS:
    r = laura[laura.fips == f].iloc[0]
    rows.append(
        f"{r.county} & {fmt_int(r.wua_scaled)} & "
        f"{r.reconstruction_capacity:.1f} & {fmt_int(r.recovery_potential_months)} \\\\"
    )
main_tex = (
    "\\begin{table}[htb!]\n\\centering\n"
    "\\caption{Modelled damage, construction capacity and recovery potential for\n"
    "four Louisiana parishes affected by Hurricane Laura (2020). Weighted units\n"
    "affected is the multi-hazard-scaled repair demand,\n"
    "$\\sum_k \\mathrm{units}_{\\mathrm{DS}k} \\times \\tau_k$ with\n"
    "$\\tau = (1, 1, 3, 6)$ months for damage states DS1--DS4. Construction\n"
    "capacity is the average monthly building-permit count. Recovery potential is\n"
    "the simulated time to work off the repair demand at that capacity. The full\n"
    "table for all damaged counties is given in the SI.}\n"
    "\\label{tab:laura_counties}\n"
    "\\begin{tabular}{lrrr}\n\\hline\n"
    "Parish (La.) & Weighted units & Capacity & Recovery potential \\\\\n"
    " & affected & (permits/month) & (months) \\\\\n\\hline\n"
    + "\n".join(rows)
    + "\n\\hline\n\\end{tabular}\n\\end{table}\n"
)
(OUT_DIR / "table_laura_main.tex").write_text(main_tex)

# ---- SI table: all damaged counties
si_rows = []
for _, r in laura.iterrows():
    rec = (
        fmt_int(r.recovery_potential_months)
        if np.isfinite(r.recovery_potential_months)
        else "--"
    )
    cap = f"{r.reconstruction_capacity:.1f}" if r.reconstruction_capacity > 0 else "0.0"
    si_rows.append(
        f"{r.county} & {r.state} & {fmt_int(r.wua_scaled)} & {fmt_int(r.wua_raw)} & "
        f"{r.repair_cost_sum_scaled / 1e6:,.1f} & {cap} & {rec} \\\\"
    )
si_tex = (
    "\\begin{table}[htb!]\n\\centering\n"
    "\\caption{All counties with modelled damage from Hurricane Laura (2020),\n"
    "sorted by scaled weighted units affected (WUA). Raw WUA uses wind-only\n"
    "damage; scaled WUA includes the rain and surge scaling. Repair cost is the\n"
    "scaled residential structural repair cost. Counties with zero permit\n"
    "capacity have undefined recovery potential (--).}\n"
    "\\label{tab:laura_counties_SI}\n"
    "\\begin{tabular}{llrrrrr}\n\\hline\n"
    "County & State & WUA & WUA & Repair cost & Capacity & Recovery \\\\\n"
    " & & (scaled) & (raw) & (million USD) & (permits/mo) & (months) \\\\\n\\hline\n"
    + "\n".join(si_rows)
    + "\n\\hline\n\\end{tabular}\n\\end{table}\n"
)
(OUT_DIR / "table_laura_SI.tex").write_text(si_tex)
laura[
    [
        "fips", "county", "state", "wua_scaled", "wua_raw",
        "repair_cost_sum_raw", "repair_cost_sum_scaled",
        "reconstruction_capacity", "recovery_potential_months",
    ]
].to_csv(OUT_DIR / "table_laura_SI.csv", index=False)


### FEMA-primary damage evaluation — `table_damage_distribution.tex`

In [ ]:
# ==================================================== damage distribution
# FEMA-primary evaluation of the modelled damage distribution.
# For each storm, three statistics against the FEMA-verified real property
# damage of owner households (OpenFEMA HousingAssistanceOwners, all declared
# counties, aggregated by fetch_openfema_damage.py):
#   1. within-storm Spearman of county WUA (scaled) vs FEMA damage over the
#      counties present in both sources (ranking skill);
#   2. detection share: fraction of the storm's total FEMA-verified damage
#      that falls in counties with modelled damage (coverage skill);
#   3. SI robustness: the same Spearman after normalising both sides by the
#      county's exposure units (damage intensity, removes county size).
# The Huang et al. (2025) building sample (n_changed) is the secondary,
# building-level anchor. Spearman is invariant to within-storm scaling, so
# correlations on county values equal correlations on within-storm shares.
huang = pd.read_csv(HUANG_CSV)
huang["GEOID"] = huang["GEOID"].astype(str).str.zfill(5)
fema = None
if FEMA_CSV.exists():
    fema = pd.read_csv(FEMA_CSV, dtype={"fips": str})
    fema["fips"] = fema["fips"].str.zfill(5)
else:
    print(f"note: {FEMA_CSV} missing - run fetch_openfema_damage.py; "
          "FEMA leg skipped")
units = pd.read_csv(REPO / "data" / "exposure_units_by_county.csv",
                    dtype={"fips": str})
units["fips"] = units["fips"].str.zfill(5)

MIN_N = 4  # smallest overlap for which a per-storm rank correlation is shown
rows, norm_rows, pooled_h, pooled_f = [], [], [], []
for atcf, (name, year) in sorted(STORMS.items(), key=lambda kv: kv[1][1]):
    df = load_event(atcf)
    df = df[df["wua_scaled"] > 0] if df is not None else None

    # ---- FEMA leg (primary)
    n_f, rho_f, p_f, det = 0, np.nan, np.nan, np.nan
    n_n, rho_n, p_n = 0, np.nan, np.nan
    f = (fema.loc[fema.atcf_id == atcf, ["fips", "fema_total_damage"]]
         if fema is not None else pd.DataFrame())
    if len(f):
        total_fema = f["fema_total_damage"].sum()
        if df is None:
            det = 0.0
        else:
            ov = df.merge(f, on="fips", how="inner")
            n_f = len(ov)
            det = ov["fema_total_damage"].sum() / total_fema
            if n_f >= MIN_N:
                rho_f, p_f = spearmanr(ov["wua_scaled"],
                                       ov["fema_total_damage"])
            if n_f >= 2:
                pooled_f.append(ov.assign(
                    sm=ov["wua_scaled"] / ov["wua_scaled"].sum(),
                    so=ov["fema_total_damage"] / ov["fema_total_damage"].sum()))
            # normalised (per exposure unit), SI robustness check
            ovn = ov.merge(units, on="fips", how="inner")
            n_n = len(ovn)
            if n_n >= MIN_N:
                rho_n, p_n = spearmanr(
                    ovn["wua_scaled"] / ovn["exposure_units"],
                    ovn["fema_total_damage"] / ovn["exposure_units"])

    # ---- Huang leg (secondary)
    n_h, rho_h, p_h = 0, np.nan, np.nan
    if df is not None:
        h = huang.loc[huang.atcf_id == atcf, ["GEOID", "n_changed"]].rename(
            columns={"GEOID": "fips"})
        ovh = df.merge(h, on="fips", how="inner")
        n_h = len(ovh)
        if n_h >= MIN_N:
            rho_h, p_h = spearmanr(ovh["wua_scaled"], ovh["n_changed"])
        if n_h >= 2:
            pooled_h.append(ovh.assign(
                sm=ovh["wua_scaled"] / ovh["wua_scaled"].sum(),
                so=ovh["n_changed"] / ovh["n_changed"].sum()))

    if len(f) == 0 and n_h == 0:
        continue
    rows.append(dict(storm=name, year=year, atcf=atcf,
                     n_fema=n_f, rho_fema=rho_f, p_fema=p_f, detection=det,
                     n_huang=n_h, rho_huang=rho_h, p_huang=p_h))
    norm_rows.append(dict(storm=name, year=year, n_raw=n_f, rho_raw=rho_f,
                          p_raw=p_f, n_norm=n_n, rho_norm=rho_n, p_norm=p_n))

dist = pd.DataFrame(rows)


def _pooled(pool):
    if not pool:
        return 0, np.nan, np.nan
    d = pd.concat(pool, ignore_index=True)
    rho, p = spearmanr(d["sm"], d["so"])
    return len(d), rho, p


np_f, rp_f, pp_f = _pooled(pooled_f)
np_h, rp_h, pp_h = _pooled(pooled_h)
dist = pd.concat([dist, pd.DataFrame([dict(
    storm="Pooled (within-storm shares)", year="", atcf="",
    n_fema=np_f, rho_fema=rp_f, p_fema=pp_f, detection=np.nan,
    n_huang=np_h, rho_huang=rp_h, p_huang=pp_h)])], ignore_index=True)
dist.to_csv(OUT_DIR / "table_damage_distribution.csv", index=False)


def _p(p):
    return "$<0.001$" if p < 0.001 else f"{p:.3f}"


def _cell(n, rho, p):
    if n == 0:
        return "-- & -- & --"
    if not np.isfinite(rho):
        return f"{int(n)} & -- & --"
    if abs(rho) == 1.0 and n < 6:
        # scipy's t-approximation returns p=0 at |rho|=1; not meaningful at
        # this sample size, so suppress the p-value
        return f"{int(n)} & ${rho:+.2f}$ & --"
    return f"{int(n)} & ${rho:+.2f}$ & {_p(p)}"


tex_rows = []
for _, r in dist.iterrows():
    label = f"{r.storm} ({r.year})" if r.year != "" else r.storm
    det = f"{100 * r.detection:.0f}\\%" if np.isfinite(r.detection) else "--"
    tex_rows.append(
        f"{label} & {_cell(r.n_fema, r.rho_fema, r.p_fema)} & {det} & "
        f"{_cell(r.n_huang, r.rho_huang, r.p_huang)} \\\\"
    )
dist_tex = (
    "\\begin{table}[htb!]\n\\centering\n"
    "\\caption{Evaluation of the modelled damage distribution against\n"
    "FEMA-verified housing damage (primary) and the Huang et al.\\ (2025)\n"
    "building sample (secondary). For each storm, the scaled weighted units\n"
    "affected (WUA) per county is rank-correlated (Spearman $r_s$, exact\n"
    "two-sided $p$) with the FEMA-verified real property damage of owner\n"
    "households from the OpenFEMA Individual Assistance records, over the\n"
    "counties present in both sources ($n$); correlations are shown for\n"
    "$n \\geq 4$. Detection is the share of the storm's total FEMA-verified\n"
    "damage, summed over all declared counties, that falls in counties with\n"
    "modelled damage. The Huang columns repeat the comparison against the\n"
    "number of assessed damaged buildings per county. The pooled row\n"
    "correlates within-storm shares across all matched county-storm pairs.\n"
    "Hermine produced no modelled damage (detection 0\\%). Nate is not part\n"
    "of the analysis (no reconstructed wind field).}\n"
    "\\label{tab:damage_distribution}\n"
    "\\begin{tabular}{lrrrrrrr}\n\\hline\n"
    " & \\multicolumn{4}{c}{FEMA IA verified damage} & "
    "\\multicolumn{3}{c}{Huang et al.\\ building sample} \\\\\n"
    "Storm & $n$ & $r_s$ & $p$ & Detection & $n$ & $r_s$ & $p$ \\\\\n\\hline\n"
    + "\n".join(tex_rows)
    + "\n\\hline\n\\end{tabular}\n\\end{table}\n"
)
(OUT_DIR / "table_damage_distribution.tex").write_text(dist_tex)


### Scaling robustness — `table_scaling_robustness.tex`

In [ ]:
# ---- SI: scaling robustness (wind-only vs scaled WUA against FEMA)
# Detection is identical by construction (the scaling is multiplicative on
# wind damage and cannot create damage where wind produced none), so only
# the rank correlations are tabulated.
sc_rows, sc_pool_r, sc_pool_s = [], [], []
for atcf, (name, year) in sorted(STORMS.items(), key=lambda kv: kv[1][1]):
    if atcf in ("AL092021", "AL092022"):  # wind-only runs: raw == scaled
        continue
    df = load_event(atcf)
    if df is None or fema is None:
        continue
    df = df[df["wua_scaled"] > 0]
    f = fema.loc[fema.atcf_id == atcf, ["fips", "fema_total_damage"]]
    ov = df.merge(f, on="fips", how="inner")
    if len(ov) < MIN_N:
        continue
    rr, pr = spearmanr(ov["wua_raw"], ov["fema_total_damage"])
    rs, ps = spearmanr(ov["wua_scaled"], ov["fema_total_damage"])
    sc_rows.append(dict(storm=name, year=year, n=len(ov),
                        rho_raw=rr, p_raw=pr, rho_scaled=rs, p_scaled=ps))
    sc_pool_r.append(ov.assign(sm=ov["wua_raw"] / ov["wua_raw"].sum(),
                               so=ov["fema_total_damage"] / ov["fema_total_damage"].sum()))
    sc_pool_s.append(ov.assign(sm=ov["wua_scaled"] / ov["wua_scaled"].sum(),
                               so=ov["fema_total_damage"] / ov["fema_total_damage"].sum()))
nr, rr_p, pr_p = _pooled(sc_pool_r)
ns, rs_p, ps_p = _pooled(sc_pool_s)
scal = pd.DataFrame(sc_rows)
scal = pd.concat([scal, pd.DataFrame([dict(
    storm="Pooled (within-storm shares)", year="", n=nr,
    rho_raw=rr_p, p_raw=pr_p, rho_scaled=rs_p, p_scaled=ps_p)])],
    ignore_index=True)
scal.to_csv(OUT_DIR / "table_scaling_robustness.csv", index=False)
tex_rows = []
for _, r in scal.iterrows():
    label = f"{r.storm} ({r.year})" if r.year != "" else r.storm
    tex_rows.append(
        f"{label} & {int(r.n)} & ${r.rho_raw:+.2f}$ & {_p(r.p_raw)} & "
        f"${r.rho_scaled:+.2f}$ & {_p(r.p_scaled)} \\\\"
    )
scal_tex = (
    "\\begin{table}[htb!]\n\\centering\n"
    "\\caption{Insensitivity of the damage evaluation to the rain and surge\n"
    "scaling. Spearman rank correlation of county weighted units affected\n"
    "with the FEMA-verified housing damage, computed with wind-only (raw)\n"
    "and with multi-hazard-scaled damage, for the storms with rain and\n"
    "surge covariates. The detection shares of\n"
    "table~\\ref{tab:damage_distribution} are identical for raw and scaled\n"
    "damage by construction, because the scaling is multiplicative on wind\n"
    "damage and cannot create damage in counties without wind damage.}\n"
    "\\label{tab:scaling_robustness}\n"
    "\\begin{tabular}{lrrrrr}\n\\hline\n"
    " & & \\multicolumn{2}{c}{Wind-only (raw)} & "
    "\\multicolumn{2}{c}{Scaled} \\\\\n"
    "Storm & $n$ & $r_s$ & $p$ & $r_s$ & $p$ \\\\\n\\hline\n"
    + "\n".join(tex_rows)
    + "\n\\hline\n\\end{tabular}\n\\end{table}\n"
)
(OUT_DIR / "table_scaling_robustness.tex").write_text(scal_tex)


### Per-exposure-unit normalisation — `table_damage_normalized.tex`

In [ ]:
# ---- SI: raw vs per-unit-normalised FEMA correlations
norm = pd.DataFrame(norm_rows)
norm.to_csv(OUT_DIR / "table_damage_normalized.csv", index=False)
tex_rows = []
for _, r in norm.iterrows():
    if r.n_raw == 0:
        continue
    tex_rows.append(
        f"{r.storm} ({r.year}) & {_cell(r.n_raw, r.rho_raw, r.p_raw)} & "
        f"{_cell(r.n_norm, r.rho_norm, r.p_norm)} \\\\"
    )
norm_tex = (
    "\\begin{table}[htb!]\n\\centering\n"
    "\\caption{Robustness of the FEMA damage-distribution comparison to\n"
    "county size. Left: Spearman correlation of county WUA with FEMA-verified\n"
    "housing damage (as in the main text). Right: the same correlation after\n"
    "dividing both quantities by the county's residential exposure units, so\n"
    "that damage intensities rather than totals are compared. Counties\n"
    "without exposure-unit data are dropped from the normalised comparison.}\n"
    "\\label{tab:damage_normalized}\n"
    "\\begin{tabular}{lrrrrrr}\n\\hline\n"
    " & \\multicolumn{3}{c}{County totals} & "
    "\\multicolumn{3}{c}{Per exposure unit} \\\\\n"
    "Storm & $n$ & $r_s$ & $p$ & $n$ & $r_s$ & $p$ \\\\\n\\hline\n"
    + "\n".join(tex_rows)
    + "\n\\hline\n\\end{tabular}\n\\end{table}\n"
)
(OUT_DIR / "table_damage_normalized.tex").write_text(norm_tex)


### Huang threshold sensitivity — `table_huang_thresholds.tex`

In [ ]:
# ================================================================ Huang table
joined = huang.merge(
    recovery, left_on=["GEOID", "atcf_id"], right_on=["fips", "event_name"],
    how="inner",
)

rows = []
for thr in [0, 10, 15, 30]:
    sub = joined[joined["n_changed"] >= thr]
    a = sub.dropna(subset=["reconstruction_capacity", "rebuild_rate"])
    rho1, p1 = spearmanr(a["reconstruction_capacity"], a["rebuild_rate"])
    b = sub.dropna(subset=["recovery_potential_months", "recovery_score"])
    rho2, p2 = spearmanr(b["recovery_potential_months"], b["recovery_score"])
    rows.append(
        dict(threshold=thr, n_capacity=len(a), rho_capacity_rebuild=rho1,
             p_capacity_rebuild=p1, n_recovery=len(b),
             rho_recovery_score=rho2, p_recovery_score=p2)
    )
th = pd.DataFrame(rows)
th.to_csv(OUT_DIR / "table_huang_thresholds.csv", index=False)

tex_rows = [
    f"$\\geq {int(r.threshold)}$ & {int(r.n_capacity)} & "
    f"${r.rho_capacity_rebuild:+.2f}$ & {r.p_capacity_rebuild:.3f} & "
    f"{int(r.n_recovery)} & ${r.rho_recovery_score:+.2f}$ & {r.p_recovery_score:.3f} \\\\"
    for _, r in th.iterrows()
]
th_tex = (
    "\\begin{table}[htb!]\n\\centering\n"
    "\\caption{Sensitivity of the comparison with Huang et al.\\ (2025) to the\n"
    "minimum county sample size (number of assessed changed buildings,\n"
    "$n_{\\mathrm{changed}}$). Spearman rank correlations, pooled across the\n"
    "matched storms, with exact two-sided p-values. Left: construction capacity\n"
    "vs observed rebuild rate (proxy validation). Right: recovery potential vs\n"
    "observed recovery score (integrated-model validation). The primary filter\n"
    "used in the main text is $n_{\\mathrm{changed}} \\geq 10$.}\n"
    "\\label{tab:huang_thresholds}\n"
    "\\begin{tabular}{lrrrrrr}\n\\hline\n"
    " & \\multicolumn{3}{c}{Capacity vs rebuild rate} & "
    "\\multicolumn{3}{c}{Recovery potential vs recovery score} \\\\\n"
    "Filter & $n$ & $r_s$ & $p$ & $n$ & $r_s$ & $p$ \\\\\n\\hline\n"
    + "\n".join(tex_rows)
    + "\n\\hline\n\\end{tabular}\n\\end{table}\n"
)
(OUT_DIR / "table_huang_thresholds.tex").write_text(th_tex)

print(f"Tables written to {OUT_DIR}")
print(dist.to_string(index=False))
print(th.to_string(index=False))


## Run summary

In [ ]:
print('=' * 65)
print('RUN SUMMARY — historical storm analysis')
print('=' * 65)

print(f'\nStorms in hazard set ({len(haz_atcf_ids)}): {haz_atcf_ids}')
print(f'Wind-only (no Dmat scaling): {hist_wind_only}')

print('\nDisplay items written:')
for fp in [FIGURES_DIR / 'hist_AL132020_4panel_focus.png',
           FIGURES_DIR / 'hist_AL142018_4panel_focus.png',
           TABLES_DIR / 'table_laura_main.tex',
           TABLES_DIR / 'table_laura_SI.tex',
           TABLES_DIR / 'table_damage_distribution.tex',
           TABLES_DIR / 'table_damage_normalized.tex',
           TABLES_DIR / 'table_scaling_robustness.tex',
           TABLES_DIR / 'table_huang_thresholds.tex']:
    status = 'ok' if fp.exists() else 'MISSING'
    print(f'  [{status:>7}] {fp.relative_to(REPO)}')

print(f'\nPipeline data:')
print(f'  Hazard HDF5  : {HAZ_HDF5}')
print(f'  Scaling NPZ  : {SCALING_NPZ_HIST}')
print(f'  Impact CSVs  : {PER_EVENT_DIR} ({len(list(PER_EVENT_DIR.glob("*.csv")))} files)')
print(f'  Recovery CSV : {RECOVERY_CSV}')
